# Grokking em Redes Neurais: O Problema da Paridade Esparsa (Sparse Parity)

Este notebook demonstra o fenômeno de **grokking** na tarefa de **Paridade Esparsa**. Diferente da aritmética modular (que se apoia em propriedades de álgebra abstrata), a paridade esparsa é um problema clássico de aprendizado de máquina intimamente ligado à física de **sistemas magnéticos desordenados e frustrados (vidros de spin)**.

### O Problema
A entrada consiste em vetores binários de tamanho $N = 10$ bits com valores em $\{-1, 1\}$. O rótulo $y \in \{-1, 1\}$ é a paridade (multiplicação ou XOR) de apenas $k = 3$ bits específicos (ex: os 3 primeiros bits):

$$y = x_0 \cdot x_1 \cdot x_2$$

Os 7 bits restantes agem como ruído puro. Como a grande maioria dos bits não tem relação com o rótulo, o gradiente inicial não dá pistas sobre quais são os $k$ bits corretos. A rede é forçada a passar por um longo platô metaestável (memorização), até que repentinamente a regularização harmônica por decaimento de pesos (*weight decay*) induz uma transição de fase de primeira ordem: a rede filtra o ruído e descobre os acoplamentos magnéticos corretos, fazendo a acurácia de validação disparar.

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# Configurar sementes para reprodutibilidade
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando o dispositivo: {device}")

### 1. Definição do Modelo MLP

Utilizamos um Perceptron Multicamadas (MLP) de uma camada oculta com ativação GELU. A inclusão de biases (`bias=True`) é fundamental para fornecer os graus de liberdade de translação necessários para resolver a paridade de forma eficiente.

In [ ]:
class ParityMLP(nn.Module):
    def __init__(self, N, hidden_dim):
        super().__init__()
        self.net = nn.Sequential( 
            nn.Linear(N, hidden_dim, bias=True),
            nn.GELU(),
            nn.Linear(hidden_dim, 2, bias=True)
        )
        
    def forward(self, x):
        return self.net(x)

### 2. Geração de Dados e Divisão de Treino/Validação

Geramos todas as $2^{10} = 1024$ configurações possíveis no hipercubo de spins. Dividimos em apenas $10\%$ para treinamento (100 exemplos) e $90\%$ para validação (924 exemplos), criando um gargalo de generalização acentuado.

In [ ]:
def generate_all_data(N, k_indices):
    grids = np.meshgrid(*[[ -1.0, 1.0 ] for _ in range(N)])
    x = torch.tensor(np.stack(grids, axis=-1).reshape(-1, N), dtype=torch.float32)
    y = x[:, k_indices].prod(dim=1)
    y = ((y + 1) / 2).long() # Mapeia {-1, 1} para classes {0, 1}
    return x, y

N = 10
k_indices = [0, 1, 2] # Paridade dos 3 primeiros bits
all_x, all_y = generate_all_data(N, k_indices)

# Divisão de treino (10%) e validação (90%)
indices = np.random.permutation(len(all_x))
train_split = 100
train_idx = indices[:train_split]
val_idx = indices[train_split:]

x_train, y_train = all_x[train_idx], all_y[train_idx]
x_val, y_val = all_x[val_idx], all_y[val_idx]

print(f"Exemplos de Treino: {len(x_train)}")
print(f"Exemplos de Validação: {len(x_val)}")

### 3. Treinamento da Rede Neural

Treinamos com o otimizador AdamW e decaimento de pesos $\lambda = 0.5$. Monitoramos o comportamento da perda, acurácia e norma $L_2$ dos pesos ao longo das épocas.

In [ ]:
model = ParityMLP(N, hidden_dim=1000).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.5)

epochs = 8000
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
    'weight_l2': []
}

x_train, y_train = x_train.to(device), y_train.to(device)
x_val, y_val = x_val.to(device), y_val.to(device)

print("Iniciando o treinamento...")
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    logits = model(x_train)
    loss = criterion(logits, y_train)
    loss.backward()
    optimizer.step()
    
    model.eval()
    with torch.no_grad():
        train_acc = (logits.argmax(dim=-1) == y_train).float().mean().item()
        val_logits = model(x_val)
        val_loss = criterion(val_logits, y_val).item()
        val_acc = (val_logits.argmax(dim=-1) == y_val).float().mean().item()
        w_l2 = sum(p.pow(2).sum().item() for p in model.parameters())
        
    history['train_loss'].append(loss.item())
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['weight_l2'].append(w_l2)
    
    if (epoch + 1) % 1000 == 0 or epoch == 0:
        print(f"Época {epoch+1:5d}/{epochs} | "
              f"Perda Treino: {loss.item():.4f} | Acc Treino: {train_acc*100:6.2f}% | "
              f"Acc Val: {val_acc*100:6.2f}% | Norma L2: {w_l2:.3f}")
        
        if train_acc > 0.999 and val_acc > 0.985:
            print(f"\nGrokking atingido precocemente na época {epoch+1}!")
            break

### 4. Resultados Gráficos: A Assinatura do Grokking

Visualização das curvas de perda, acurácia e norma dos pesos $L_2$. Observe o longo platô metaestável (memorização) onde a perda de validação sobe e a acurácia de validação fica estagnada próxima a $50\%$. Em seguida, as forças de restrição do *weight decay* encolhem os pesos (norma L2 diminui), forçando o modelo a colapsar para a estrutura generalizadora simples (XOR) e disparando a acurácia de validação para $100\%$.

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(6.5, 9.0), sharex=True)

# 1. Perda
ax1.plot(history['train_loss'], label='Treino', color='blue', alpha=0.7)
ax1.plot(history['val_loss'], label='Validação', color='red', alpha=0.7)
ax1.set_yscale('log')
ax1.set_title('Perda vs Épocas')
ax1.set_ylabel('Perda (Log)')
ax1.legend()
ax1.grid(True, which='both', ls='--')

# 2. Acurácia
ax2.plot(history['train_acc'], label='Treino', color='blue', alpha=0.7)
ax2.plot(history['val_acc'], label='Validação', color='red', alpha=0.7)
ax2.set_title('Acurácia vs Épocas')
ax2.set_ylabel('Acurácia (%)')
ax2.legend()
ax2.grid(True, ls='--')

# 3. Norma L2
ax3.plot(history['weight_l2'], label='Norma L2 (w)', color='purple', alpha=0.7)
ax3.set_title('Norma dos Pesos L2 vs Épocas')
ax3.set_xlabel('Épocas')
ax3.set_ylabel('Norma L2')
ax3.legend()
ax3.grid(True, ls='--')

plt.suptitle('Grokking no Problema de Paridade Esparsa (N=10, k=3)', fontsize=12, y=0.99)
plt.tight_layout()

# Salvar o gráfico
import os
os.makedirs('../paper', exist_ok=True)
plt.savefig('../paper/grokking_parity.png', dpi=300)
plt.show()